In [1]:
import os
import json
import datetime
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text
    
engine = create_engine('mysql+pymysql://root@localhost:3306/music_development')
music = engine.connect()
data_path = '../data/'

In [2]:
def scan_mp4_files(folder_path):
    """Scan folder for MP4 files and return sorted list"""
    mp4_files = []
    
    try:
        for file in os.listdir(folder_path):
            if file.lower().endswith('.mp4'):
                mp4_files.append(file)
        
        # Sort alphabetically
        mp4_files.sort()
        
    except Exception as e:
        print(f"Error scanning folder: {e}")
    
    return mp4_files

In [3]:
def analyze_filenames_complete(mp4_files):
    """Extract song titles and artists from MP4 filenames into DataFrame"""
    
    song_titles = []
    artists = []
    filenames = []
    
    for filename in mp4_files:
        # Remove .mp4 extension
        name_without_ext = filename[:-4]
        
        # Extract song title and artist
        if '_' in name_without_ext:
            parts = name_without_ext.split('_')
            song_title = parts[0].strip()
            artist = ' '.join(parts[1:]).strip() if len(parts) > 1 else ''
        else:
            song_title = name_without_ext
            artist = ''
        
        song_titles.append(song_title)
        artists.append(artist)
        filenames.append(filename)
    
    # Create DataFrame with multiple columns
    return pd.DataFrame({
        'song_title': song_titles,
        'artist': artists,
        'filename': filenames
    })

In [4]:
# Set your Lyrics folder path - CHANGE THIS TO YOUR MP4 FOLDER
LYRICS_FOLDER = r"C:\Users\PC1\OneDrive\A5\Data\Videos\AWS"   #chgfld  
mp4_files = scan_mp4_files(LYRICS_FOLDER)

# Use the enhanced function
df = analyze_filenames_complete(mp4_files)
# print("\n📊 Complete DataFrame from filenames:")
# print(df.head())

# Get songs from database
sql = 'SELECT * FROM songs'
df_songs = pd.read_sql(sql, music)

col = ['id','name','youtube_code']
df_songs_3col = df_songs[col]
df_songs_3col.columns

Index(['id', 'name', 'youtube_code'], dtype='object')

In [24]:
# Check for matching columns
print("\n🔍 Available columns for merging:")
# print(f"df columns: {df.columns.tolist()}")
# print(f"df_songs columns: {df_songs.columns.tolist()}")

# Try different merge strategies
# Option 1: Merge on song title
if 'title' in df_songs.columns:
    df_merge = pd.merge(df, df_songs_3col, left_on='song_title', right_on='title', how='left')
    print("\n✅ Merged on song title:")
    print(df_merge[['song_title', 'artist', 'title', 'filename']].head())
    
# Option 2: If there's a name column
elif 'name' in df_songs.columns:
    df_merge = pd.merge(df, df_songs_3col, left_on='song_title', right_on='name', how='left')
    print("\n✅ Merged on name:")
    df_merge.drop(columns=['name'],inplace=True)
    print(df_merge[['song_title', 'artist']].head())

# Save to CSV for inspection
df_merge.to_csv('merged_songs.csv', index=False, encoding='utf-8')
print("\n💾 Saved merged data to 'merged_songs.csv'")


🔍 Available columns for merging:

✅ Merged on name:
                          song_title        artist
0  (Everything I Do) I Do It For You   Bryan Adams
1             A Change Is Gonna Come     Sam Cooke
2                  A Day in the Life   Beatles The
3                   Against the Wind     Bob Seger
4                       All Too Well  Taylor Swift

💾 Saved merged data to 'merged_songs.csv'
